# Chapter 5.8: Constrained Decoding — JSON, Regex, Grammar

## Learning Objectives

By the end of this notebook, you will:

1. **Understand why constrained decoding matters** for structured LLM output
2. **Learn how JSON schema maps to finite state machines**
3. **Master regex-guided generation** and its implementation
4. **Understand context-free grammar** support for complex formats
5. **Explore jump-forward decoding** optimization
6. **See how XGrammar and Outlines** integrate with SGLang

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.8_constrained_decoding.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.8_constrained_decoding.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Constrained Decoding?

LLMs generate text token by token. Without constraints, they can produce:
- Invalid JSON (missing commas, unclosed brackets)
- Wrong types (string instead of number)
- Missing required fields
- Unexpected formats

**Constrained decoding** guarantees the output follows a specified format by **masking invalid tokens** at each generation step.

```
Without constraint:   {"name": "Alice", age: 30}    <- Invalid JSON!
With JSON constraint:  {"name": "Alice", "age": 30}  <- Always valid!

How it works:
Step 1: Generate '{'      <- Only '{' is valid
Step 2: Generate '"'      <- Only '"' is valid (start of key)
Step 3: Generate 'name'   <- Any string token valid
Step 4: Generate '"'      <- Only '"' is valid (end of key)
Step 5: Generate ':'      <- Only ':' is valid
...
```

## 2. Finite State Machines for Constrained Decoding

The core idea: represent the constraint as a **Finite State Machine (FSM)** and use it to mask invalid tokens.

```
JSON Object FSM (simplified):

    ┌─────┐    {     ┌─────┐    "     ┌─────┐
    │START │───────→ │OBJ  │───────→ │KEY  │
    └─────┘          └─────┘          └──┬──┘
                        ↑                │ "
                        │     :          ▼
                        │  ┌─────┐   ┌─────┐
                        │  │VALUE│←──│COLON│
                        │  └──┬──┘   └─────┘
                        │     │
                        │  ,  │ value
                        └─────┘
                           │
                           │ }
                           ▼
                        ┌─────┐
                        │ END │
                        └─────┘
```

At each state, only certain tokens are valid. We create a **token mask** that sets the probability of invalid tokens to `-inf`.

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(14, 6))ax.set_xlim(-0.5, 14)ax.set_ylim(-0.5, 6)ax.axis('off')fig.patch.set_facecolor('white')ax.text(7, 5.5, 'FSM-Guided JSON Generation', fontsize=14, fontweight='bold',        ha='center', color=DARK_GRAY)# Statesstates_data = [    (0.5, 3, "START", DARK_GRAY),    (2.5, 3, '{', BLUE),    (4.5, 3, '"key"', GREEN),    (6.5, 3, ':', ORANGE),    (8.5, 3, '"value"', RED),    (10.5, 3, ',  or  }', PURPLE),    (12.5, 3, 'END', DARK_GRAY),]for x, y, label, color in states_data:    circle = plt.Circle((x, y), 0.6, facecolor=color, edgecolor='white', lw=2, alpha=0.85)    ax.add_patch(circle)    ax.text(x, y, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')# Arrows between statesfor i in range(len(states_data)-1):    x1 = states_data[i][0] + 0.6    x2 = states_data[i+1][0] - 0.6    ax.annotate('', xy=(x2, 3), xytext=(x1, 3),                arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=1.5))# Loop back from , to "key"ax.annotate('', xy=(4.5, 3.7), xytext=(10.2, 3.7),            arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.5,                           connectionstyle='arc3,rad=0.4'))ax.text(7.5, 4.8, 'if comma: add more fields', fontsize=8, color=ORANGE, ha='center')# Token constraintsconstraints = [    (0.5, 1.5, 'Valid tokens:\nonly "{"'),    (2.5, 1.5, 'Valid tokens:\nonly "\""'),    (4.5, 1.5, 'Valid tokens:\nany string'),    (6.5, 1.5, 'Valid tokens:\nonly ":"'),    (8.5, 1.5, 'Valid tokens:\nany value'),    (10.5, 1.5, 'Valid tokens:\n"," or "}"'),]for x, y, text in constraints:    ax.text(x, y, text, ha='center', va='center', fontsize=7, color=DARK_GRAY,            bbox=dict(boxstyle='round,pad=0.2', facecolor='#F0F0F0', alpha=0.6))    ax.annotate('', xy=(x, 2.4), xytext=(x, 1.9),                arrowprops=dict(arrowstyle='->', color=GRAY, lw=0.8))plt.tight_layout()plt.savefig("fsm_json_generation.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Implement a simple regex-guided FSM

from typing import Set, Dict, List, Tuple, Optional
from enum import Enum
import re

class FSMState:
    """A state in a Finite State Machine."""
    def __init__(self, name: str, is_final: bool = False):
        self.name = name
        self.is_final = is_final
        self.transitions: Dict[str, 'FSMState'] = {}  # char -> next_state
    
    def add_transition(self, char: str, next_state: 'FSMState'):
        self.transitions[char] = next_state
    
    def get_valid_chars(self) -> Set[str]:
        return set(self.transitions.keys())
    
    def __repr__(self):
        final = " (FINAL)" if self.is_final else ""
        return f"State({self.name}{final})"


class SimpleRegexFSM:
    """A simple FSM for regex-guided token generation."""
    
    def __init__(self):
        self.states: Dict[str, FSMState] = {}
        self.start_state: Optional[FSMState] = None
        self.current_state: Optional[FSMState] = None
    
    def add_state(self, name: str, is_final: bool = False) -> FSMState:
        state = FSMState(name, is_final)
        self.states[name] = state
        if self.start_state is None:
            self.start_state = state
            self.current_state = state
        return state
    
    def add_transition(self, from_name: str, char: str, to_name: str):
        self.states[from_name].add_transition(char, self.states[to_name])
    
    def get_valid_tokens(self, vocabulary: Dict[str, int]) -> Set[int]:
        """Get token IDs that are valid from current state."""
        valid = set()
        for token_str, token_id in vocabulary.items():
            if self.can_accept(token_str):
                valid.add(token_id)
        return valid
    
    def can_accept(self, token_str: str) -> bool:
        """Check if a token string is valid from current state."""
        state = self.current_state
        for char in token_str:
            if char not in state.transitions:
                return False
            state = state.transitions[char]
        return True
    
    def advance(self, token_str: str) -> bool:
        """Advance FSM by consuming a token."""
        for char in token_str:
            if char not in self.current_state.transitions:
                return False
            self.current_state = self.current_state.transitions[char]
        return True
    
    def reset(self):
        self.current_state = self.start_state


# Build FSM for simple integer pattern: [0-9]+
def build_integer_fsm() -> SimpleRegexFSM:
    fsm = SimpleRegexFSM()
    
    start = fsm.add_state("start")
    digit = fsm.add_state("digit", is_final=True)
    
    # Start -> digit (first digit required)
    for d in "0123456789":
        fsm.add_transition("start", d, "digit")
    
    # Digit -> digit (more digits optional)
    for d in "0123456789":
        fsm.add_transition("digit", d, "digit")
    
    return fsm

# Demo
fsm = build_integer_fsm()

# Simple vocabulary
vocab = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "42": 10, "100": 11,  # Multi-char tokens
    "hello": 12, "world": 13, " ": 14, ".": 15,
}

print("Integer FSM Demo")
print("=" * 50)

# Show valid tokens at each step
fsm.reset()
print(f"State: {fsm.current_state}")
valid = fsm.get_valid_tokens(vocab)
valid_tokens = [t for t, tid in vocab.items() if tid in valid]
print(f"Valid tokens: {valid_tokens}")

# Generate "42"
fsm.advance("4")
print(f"\nAfter '4': state={fsm.current_state}")
valid = fsm.get_valid_tokens(vocab)
valid_tokens = [t for t, tid in vocab.items() if tid in valid]
print(f"Valid tokens: {valid_tokens}")

fsm.advance("2")
print(f"\nAfter '42': state={fsm.current_state}")
print(f"Is final state: {fsm.current_state.is_final}")

## 3. JSON Schema to FSM

For JSON schema-guided generation:

```python
# JSON Schema
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
        "city": {"type": "string"}
    },
    "required": ["name", "age"]
}

# This gets compiled to a regex:
# \{\s*"name"\s*:\s*"[^"]*"\s*,\s*"age"\s*:\s*[0-9]+\s*(,\s*"city"\s*:\s*"[^"]*")?\s*\}

# Which is then converted to an FSM for token masking
```

In [ ]:
# Demonstrate JSON schema to regex conversion

class JSONSchemaToRegex:
    """Converts JSON schema to regex pattern."""
    
    def __init__(self):
        self.ws = r'\s*'  # Optional whitespace
    
    def convert(self, schema: dict) -> str:
        """Convert a JSON schema to a regex pattern."""
        schema_type = schema.get("type", "string")
        
        if schema_type == "string":
            return self._string_pattern(schema)
        elif schema_type == "integer":
            return self._integer_pattern(schema)
        elif schema_type == "number":
            return self._number_pattern(schema)
        elif schema_type == "boolean":
            return r'(true|false)'
        elif schema_type == "null":
            return r'null'
        elif schema_type == "array":
            return self._array_pattern(schema)
        elif schema_type == "object":
            return self._object_pattern(schema)
        else:
            return r'.*'
    
    def _string_pattern(self, schema):
        if "enum" in schema:
            options = "|".join(f'"{v}"' for v in schema["enum"])
            return f'({options})'
        return r'"[^"]*"'
    
    def _integer_pattern(self, schema):
        return r'-?[0-9]+'
    
    def _number_pattern(self, schema):
        return r'-?[0-9]+(\.[0-9]+)?'
    
    def _array_pattern(self, schema):
        items = schema.get("items", {"type": "string"})
        item_pattern = self.convert(items)
        return f'\\[{self.ws}({item_pattern}({self.ws},{self.ws}{item_pattern})*)?{self.ws}\\]'
    
    def _object_pattern(self, schema):
        properties = schema.get("properties", {})
        required = set(schema.get("required", []))
        
        parts = []
        for prop_name, prop_schema in properties.items():
            prop_pattern = self.convert(prop_schema)
            field = f'"{prop_name}"{self.ws}:{self.ws}{prop_pattern}'
            if prop_name not in required:
                field = f'({field})?'
            parts.append(field)
        
        fields = f'{self.ws},{self.ws}'.join(parts)
        return f'\\{{{self.ws}{fields}{self.ws}\\}}'

# Demo
converter = JSONSchemaToRegex()

schemas = [
    ("Simple string", {"type": "string"}),
    ("Integer", {"type": "integer"}),
    ("Enum", {"type": "string", "enum": ["red", "green", "blue"]}),
    ("Object", {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "age": {"type": "integer"},
        },
        "required": ["name", "age"]
    }),
]

print("JSON Schema -> Regex Conversion")
print("=" * 70)
for name, schema in schemas:
    regex = converter.convert(schema)
    print(f"\n  {name}:")
    print(f"    Schema: {schema}")
    print(f"    Regex:  {regex}")

## 4. Jump-Forward Decoding

A key optimization in SGLang: **jump-forward decoding**.

When the FSM dictates that only one sequence of tokens is valid (a "forced" path), we can **skip ahead** without running the model.

```
Example: Generating JSON with schema {"name": string, "age": integer}

Standard decoding:                    Jump-forward decoding:
Step 1: Generate '{'   (1 option)     Step 1: Jump '{"name": '  (forced!)
Step 2: Generate '"'   (1 option)     Step 2: Generate name value (free)
Step 3: Generate 'n'   (1 option)     Step 3: Jump '", "age": ' (forced!)
Step 4: Generate 'a'   (1 option)     Step 4: Generate age value (free)
Step 5: Generate 'm'   (1 option)     Step 5: Jump '}'          (forced!)
Step 6: Generate 'e'   (1 option)     
Step 7: Generate '"'   (1 option)     5 steps instead of 15+!
Step 8: Generate ':'   (1 option)
Step 9: Generate ' '   (1 option)
Step 10: Generate '"'  (1 option)
Step 11-N: Generate name (free)
...
```

**Speedup**: Significant for structured outputs where most characters are forced.

In [ ]:
# Demonstrate jump-forward decoding

class JumpForwardDecoder:
    """Demonstrates jump-forward decoding optimization."""
    
    def __init__(self):
        self.steps_standard = 0
        self.steps_jumpforward = 0
    
    def simulate_json_generation(self, schema, values):
        """Simulate generating JSON with and without jump-forward."""
        
        # Build the forced structure
        properties = schema.get("properties", {})
        
        # Standard decoding: every character is a step
        standard_steps = []
        json_str = '{'
        standard_steps.append(('{', 'forced'))
        
        for i, (prop_name, prop_schema) in enumerate(properties.items()):
            if i > 0:
                json_str += ', '
                standard_steps.append((',', 'forced'))
                standard_steps.append((' ', 'forced'))
            
            # Key
            json_str += f'"{prop_name}": '
            for c in f'"{prop_name}": ':
                standard_steps.append((c, 'forced'))
            
            # Value
            value = values.get(prop_name, '""')
            json_str += str(value)
            for c in str(value):
                standard_steps.append((c, 'free' if prop_schema['type'] in ['string', 'integer'] else 'forced'))
        
        json_str += '}'
        standard_steps.append(('}', 'forced'))
        
        # Jump-forward decoding: group forced sequences
        jump_steps = []
        current_forced = ""
        
        for char, step_type in standard_steps:
            if step_type == 'forced':
                current_forced += char
            else:
                if current_forced:
                    jump_steps.append((current_forced, 'jump'))
                    current_forced = ""
                jump_steps.append((char, 'generate'))
        
        if current_forced:
            jump_steps.append((current_forced, 'jump'))
        
        return standard_steps, jump_steps, json_str

# Demo
decoder = JumpForwardDecoder()

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
        "city": {"type": "string"},
    }
}

values = {"name": '"Alice"', "age": "30", "city": '"Paris"'}

standard, jumpforward, json_output = decoder.simulate_json_generation(schema, values)

print("Jump-Forward Decoding Demo")
print("=" * 70)
print(f"Output: {json_output}")

print(f"\nStandard decoding: {len(standard)} steps")
forced = sum(1 for _, t in standard if t == 'forced')
free = sum(1 for _, t in standard if t == 'free')
print(f"  Forced steps: {forced} ({forced/len(standard)*100:.0f}%)")
print(f"  Free steps: {free} ({free/len(standard)*100:.0f}%)")

print(f"\nJump-forward decoding: {len(jumpforward)} steps")
for text, step_type in jumpforward:
    label = "JUMP" if step_type == 'jump' else "GEN "
    print(f"  [{label}] {repr(text)}")

speedup = len(standard) / len(jumpforward)
print(f"\nSpeedup: {speedup:.1f}x fewer model forward passes")

## 5. XGrammar Integration

XGrammar is SGLang's default grammar backend for constrained decoding:

```python
# sglang/srt/constrained/xgrammar_backend.py (simplified)

class XGrammarBackend:
    def __init__(self, tokenizer):
        import xgrammar
        self.tokenizer = tokenizer
        self.compiler = xgrammar.GrammarCompiler(
            tokenizer_info=xgrammar.TokenizerInfo.from_huggingface(tokenizer)
        )
        self.cache = {}  # Cache compiled grammars
    
    def get_mask_for_json_schema(self, schema_str, state=None):
        """Get token mask for JSON schema constraint."""
        if schema_str not in self.cache:
            # Compile JSON schema to grammar
            compiled = self.compiler.compile_json_schema(schema_str)
            self.cache[schema_str] = compiled
        
        grammar = self.cache[schema_str]
        
        # Create or advance the matcher
        if state is None:
            matcher = xgrammar.GrammarMatcher(grammar)
        else:
            matcher = state.matcher
        
        # Get valid token mask
        token_bitmask = matcher.get_next_token_bitmask()
        # token_bitmask: tensor of shape [vocab_size] with 1 for valid, 0 for invalid
        
        return token_bitmask, matcher
    
    def advance_state(self, matcher, token_id):
        """Advance grammar state after generating a token."""
        matcher.accept_token(token_id)
```

In [ ]:
# Simulate constrained decoding with token masking

import math
import random

class ConstrainedDecoder:
    """Simulates constrained decoding with token masking."""
    
    def __init__(self, vocab: Dict[str, int]):
        self.vocab = vocab
        self.id_to_token = {v: k for k, v in vocab.items()}
    
    def generate_constrained(self, logits_fn, constraint_fn, max_tokens=50):
        """Generate tokens with constraint masking.
        
        Args:
            logits_fn: Function that returns logits for each token
            constraint_fn: Function(generated_so_far) -> set of valid token IDs
            max_tokens: Maximum tokens to generate
        """
        generated = []
        generated_text = ""
        
        for step in range(max_tokens):
            # Get model logits (simulated)
            logits = logits_fn(generated_text)
            
            # Get valid tokens from constraint
            valid_tokens = constraint_fn(generated_text)
            
            if not valid_tokens:
                break  # No valid tokens = done
            
            # Mask invalid tokens
            masked_logits = {}
            for token_id in valid_tokens:
                token_str = self.id_to_token[token_id]
                masked_logits[token_id] = logits.get(token_id, 0.0)
            
            # Sample from valid tokens
            # For simplicity, use greedy (argmax)
            best_id = max(masked_logits, key=masked_logits.get)
            best_token = self.id_to_token[best_id]
            
            generated.append(best_id)
            generated_text += best_token
        
        return generated_text, generated

# Simple vocabulary for JSON generation
json_vocab = {
    '{': 0, '}': 1, '[': 2, ']': 3, '"': 4, ':': 5, ',': 6, ' ': 7,
    'name': 8, 'age': 9, 'city': 10,
    'Alice': 11, 'Bob': 12, 'Paris': 13, 'London': 14,
    '25': 15, '30': 16, '35': 17,
    'true': 18, 'false': 19, 'null': 20,
}

# Simple constraint for {"name": "<str>", "age": <int>}
def json_constraint(generated_so_far):
    """Return valid token IDs based on what's been generated."""
    id_to_token = {v: k for k, v in json_vocab.items()}
    
    patterns = [
        ("", {'{'}),
        ('{', {'"'}),
        ('{"', {'name'}),
        ('{"name', {'"'}),
        ('{"name"', {':'}),
        ('{"name":', {' '}),
        ('{"name": ', {'"'}),
        ('{"name": "', {'Alice', 'Bob'}),
    ]
    
    for prefix, valid_tokens in patterns:
        if generated_so_far == prefix:
            return {json_vocab[t] for t in valid_tokens}
    
    # After name value
    if generated_so_far.endswith('Alice') or generated_so_far.endswith('Bob'):
        return {json_vocab['"']}
    
    if '"name": "' in generated_so_far and generated_so_far.endswith('"') and 'age' not in generated_so_far:
        return {json_vocab[',']}
    
    if generated_so_far.endswith(', '):
        return {json_vocab['"']}
    
    if generated_so_far.endswith(', "'):
        return {json_vocab['age']}
    
    if generated_so_far.endswith('age'):
        return {json_vocab['"']}
    
    if generated_so_far.endswith('age"'):
        return {json_vocab[':']}
    
    if generated_so_far.endswith('age":'):
        return {json_vocab[' ']}
    
    if generated_so_far.endswith('age": '):
        return {json_vocab['25'], json_vocab['30'], json_vocab['35']}
    
    if any(generated_so_far.endswith(str(v)) for v in ['25', '30', '35']):
        return {json_vocab['}']}
    
    if generated_so_far.endswith('}'):
        return set()  # Done!
    
    return {json_vocab[',']}  # fallback

# Simulate
decoder = ConstrainedDecoder(json_vocab)

def fake_logits(text):
    """Simulated model preferences."""
    logits = {v: random.uniform(-1, 1) for v in json_vocab.values()}
    # Bias toward Alice and 30
    logits[json_vocab['Alice']] = 2.0
    logits[json_vocab['30']] = 1.5
    return logits

random.seed(42)
result_text, result_ids = decoder.generate_constrained(
    fake_logits, json_constraint, max_tokens=20
)

print("Constrained JSON Generation")
print("=" * 50)
print(f"Generated: {result_text}")
print(f"Token IDs: {result_ids}")
print(f"Steps: {len(result_ids)}")

## 6. Context-Free Grammar Support

For more complex formats, SGLang supports **EBNF grammars**:

```ebnf
# Example: SQL SELECT statement grammar
root       ::= "SELECT " columns " FROM " table_name where_clause?
columns    ::= column (", " column)*
column     ::= [a-zA-Z_]+
table_name ::= [a-zA-Z_]+
where_clause ::= " WHERE " condition
condition  ::= column " = " value
value      ::= "'" [^']* "'" | [0-9]+
```

This enables generating:
- Valid SQL queries
- Valid Python code
- Custom DSL expressions
- Structured data in any format

## 7. Integration with SGLang Server

```python
# Using constrained decoding via the API

# Method 1: JSON Schema
response = client.chat.completions.create(
    model="llama-3.1-8b",
    messages=[{"role": "user", "content": "Generate a user profile"}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "user_profile",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "age": {"type": "integer"},
                },
                "required": ["name", "age"]
            }
        }
    }
)

# Method 2: Regex
response = client.chat.completions.create(
    model="llama-3.1-8b",
    messages=[{"role": "user", "content": "What is your phone number?"}],
    extra_body={"regex": r"\d{3}-\d{3}-\d{4}"}
)

# Method 3: SGLang DSL
@sgl.function
def extract_info(s, text):
    s += sgl.user(f"Extract info: {text}")
    s += sgl.assistant(
        sgl.gen("result", 
                regex=r'{"name": "[A-Za-z ]+", "age": [0-9]+}')
    )
```

In [ ]:
# Show the API usage patterns

print("SGLang Constrained Decoding API Examples")
print("=" * 60)

examples = [
    {
        "name": "JSON Mode (simple)",
        "api": 'response_format={"type": "json_object"}',
        "output": '{"answer": "Paris", "confidence": 0.95}',
    },
    {
        "name": "JSON Schema (strict)",
        "api": 'response_format={"type": "json_schema", "json_schema": {...}}',
        "output": '{"name": "Alice", "age": 30}',
    },
    {
        "name": "Regex constraint",
        "api": 'extra_body={"regex": r"\\d{3}-\\d{3}-\\d{4}"}',
        "output": '555-123-4567',
    },
    {
        "name": "Choice constraint",
        "api": 'sgl.select(["positive", "negative", "neutral"])',
        "output": 'positive',
    },
    {
        "name": "EBNF Grammar",
        "api": 'extra_body={"ebnf": "root ::= ..."}',
        "output": 'SELECT name, age FROM users WHERE id = 42',
    },
]

for ex in examples:
    print(f"\n  {ex['name']}:")
    print(f"    API:    {ex['api']}")
    print(f"    Output: {ex['output']}")

## 8. Summary

### Key Takeaways

1. **Constrained decoding** guarantees structured output by masking invalid tokens
2. **FSM-based approach**: Convert schema/regex to FSM, use FSM state for token masking
3. **Jump-forward decoding** skips forced tokens for significant speedup
4. **XGrammar** is the default backend (fast, supports JSON/regex/EBNF)
5. **Multiple constraint types**: JSON schema, regex, EBNF grammar, choice
6. **Integration**: Available via OpenAI API, SGLang native API, and DSL

### Next Chapter

Chapter 5.9 will explore the **SGLang Frontend DSL** — the programming language for LLMs.

## Exercises

### Exercise 1: Build a Regex FSM
Implement an FSM for the regex pattern `[A-Z][a-z]+ [A-Z][a-z]+` (first and last name).

### Exercise 2: JSON Schema Validator
Extend the JSON schema converter to handle nested objects and arrays.

### Exercise 3: Custom Grammar
Write an EBNF grammar for generating valid Python function signatures.

### Exercise 4: Performance Analysis
Calculate the theoretical speedup of jump-forward decoding for various JSON schemas with different ratios of forced vs. free tokens.

In [ ]:
# Exercise 4: Jump-forward speedup analysis

schemas = [
    ("Simple object (2 fields)", 30, 10),    # 30 forced, 10 free chars
    ("Complex object (10 fields)", 200, 50), # 200 forced, 50 free
    ("Nested object (3 levels)", 150, 30),   # 150 forced, 30 free
    ("Array of objects", 100, 100),           # 100 forced, 100 free
    ("Free-form text", 0, 500),               # 0 forced, all free
]

print("Jump-Forward Speedup Analysis")
print("=" * 70)
print(f"{'Schema':<30s} {'Forced':>8s} {'Free':>6s} {'Standard':>10s} {'JumpFwd':>9s} {'Speedup':>9s}")
print("-" * 70)

for name, forced, free in schemas:
    standard_steps = forced + free  # Every char is a step
    # Jump-forward: forced chars grouped into ~1 step each group
    # Assume ~5 forced groups on average
    num_forced_groups = max(1, forced // 20)  # rough estimate
    jumpfwd_steps = num_forced_groups + free
    
    speedup = standard_steps / max(1, jumpfwd_steps)
    print(f"{name:<30s} {forced:8d} {free:6d} {standard_steps:10d} {jumpfwd_steps:9d} {speedup:8.1f}x")